# Food Delivery ETA Estimation Using Regression-Based Machine Learning


## 1. Name and Roll Number

**Name:** Khushi Gupta                      **Roll Number:** 24csu100 
                    
**Name:** Madhavan Dixit                    **Roll Number:** 24csu118


## 2. Title of the Project

**Food Delivery ETA Estimation Using Regression-Based Machine Learning**

This project aims to predict the estimated time of arrival (ETA) for food delivery orders using various regression techniques, and to classify deliveries into speed categories (e.g., Fast / Medium / Slow) using classification techniques, based on real-world DoorDash delivery data.


## 3. Brief Description of Dataset

**Dataset:** DoorDash ETA Prediction Dataset  
**Source:** Kaggle — `dharun4772/doordash-eta-prediction` (originally from DoorDash's data science take-home challenge)

The dataset (`historical_data.csv`) contains a subset of real deliveries received by DoorDash in early 2015. Each row corresponds to one unique delivery. The goal is to predict `delivery_duration` — the total time (in seconds) between when the consumer places the order (`created_at`) and when it is actually delivered (`actual_delivery_time`).

**Key columns:**

| Column | Description |
|---|---|
| `market_id` | City/region ID where the delivery took place |
| `created_at` | Timestamp order was placed (UTC) |
| `actual_delivery_time` | Timestamp order was delivered (UTC) |
| `store_id` | Restaurant ID |
| `store_primary_category` | Cuisine type of restaurant |
| `order_protocol` | Protocol used to place the order |
| `total_items` | Number of items in the order |
| `subtotal` | Order subtotal (in cents) |
| `num_distinct_items` | Number of distinct items ordered |
| `min_item_price` / `max_item_price` | Price range of items in order |
| `total_onshift_dashers` | Dashers on shift at order time |
| `total_busy_dashers` | Dashers currently busy at order time |
| `total_outstanding_orders` | Orders outstanding in the market at order time |
| `estimated_order_place_duration` | DoorDash's own estimate of order placement time |
| `estimated_store_to_consumer_driving_duration` | DoorDash's own estimate of driving time |

**Target variable (engineered):** `delivery_duration` = `actual_delivery_time` − `created_at` (in seconds)


## Setup: Import Libraries


In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & feature selection
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')


## 4. Data Preprocessing

Steps: load data, engineer the target variable, handle missing values, encode categorical features, and derive useful time-based features.


In [ ]:
# Load dataset
# Download historical_data.csv from:
# https://www.kaggle.com/datasets/dharun4772/doordash-eta-prediction
# and place it in the same folder as this notebook.

df = pd.read_csv('historical_data.csv')
print(df.shape)
df.head()


In [ ]:
# Basic info
df.info()


In [ ]:
# Check missing values
df.isnull().sum().sort_values(ascending=False)


In [ ]:
# Engineer target variable: delivery duration in seconds
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])
df['delivery_duration'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()

# Drop rows where target could not be computed
df = df.dropna(subset=['delivery_duration'])

# Remove unrealistic durations (outliers), e.g. negative or extremely large
df = df[(df['delivery_duration'] > 0) & (df['delivery_duration'] < df['delivery_duration'].quantile(0.99))]

df[['created_at', 'actual_delivery_time', 'delivery_duration']].head()


In [ ]:
# Extract time-based features
df['order_hour'] = df['created_at'].dt.hour
df['order_day_of_week'] = df['created_at'].dt.dayofweek
df['is_weekend'] = df['order_day_of_week'].isin([5, 6]).astype(int)


In [ ]:
# Handle missing values in numeric/categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
# Encode categorical variables
le_dict = {}
for col in ['store_primary_category']:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

df.head()


## 5. Data Visualization

Explore distributions, relationships, and correlations to understand the data before modeling.


In [ ]:
# Distribution of target variable
plt.figure(figsize=(8, 5))
sns.histplot(df['delivery_duration'], bins=50, kde=True)
plt.title('Distribution of Delivery Duration (seconds)')
plt.xlabel('Delivery Duration (s)')
plt.show()


In [ ]:
# Delivery duration by hour of day
plt.figure(figsize=(10, 5))
sns.boxplot(x='order_hour', y='delivery_duration', data=df)
plt.title('Delivery Duration by Order Hour')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Correlation heatmap of numeric features
plt.figure(figsize=(12, 8))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
# Delivery duration vs total items / subtotal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x='total_items', y='delivery_duration', data=df, alpha=0.3, ax=axes[0])
axes[0].set_title('Total Items vs Delivery Duration')
sns.scatterplot(x='subtotal', y='delivery_duration', data=df, alpha=0.3, ax=axes[1])
axes[1].set_title('Subtotal vs Delivery Duration')
plt.tight_layout()
plt.show()


## 6. Feature Selection

Using **SelectKBest with f_regression** to identify the most relevant features for predicting delivery duration.


In [ ]:
# Define feature set (drop non-feature / leakage columns)
drop_cols = ['created_at', 'actual_delivery_time', 'delivery_duration', 'store_primary_category']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_cols]

X = df[feature_cols]
y = df['delivery_duration']

selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'Feature': feature_cols,
    'Score': selector.scores_
}).sort_values(by='Score', ascending=False)

feature_scores


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Score', y='Feature', data=feature_scores)
plt.title('Feature Importance Scores (f_regression)')
plt.show()

# Select top K features for modeling
top_k = 8
selected_features = feature_scores['Feature'].head(top_k).tolist()
print('Selected features:', selected_features)


## 7. Regression Analysis (3 Techniques)

We compare **Linear Regression**, **Random Forest Regressor**, and **Gradient Boosting Regressor** to predict `delivery_duration`.


In [ ]:
X = df[selected_features]
y = df['delivery_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
def evaluate_regressor(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    mae = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2 = r2_score(y_te, preds)
    print(f'--- {name} ---')
    print(f'MAE:  {mae:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'R2:   {r2:.4f}\n')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []
results.append(evaluate_regressor('Linear Regression', LinearRegression(),
                                   X_train_scaled, X_test_scaled, y_train, y_test))
results.append(evaluate_regressor('Random Forest Regressor', RandomForestRegressor(n_estimators=100, random_state=42),
                                   X_train, X_test, y_train, y_test))
results.append(evaluate_regressor('Gradient Boosting Regressor', GradientBoostingRegressor(random_state=42),
                                   X_train, X_test, y_train, y_test))

regression_results_df = pd.DataFrame(results)
regression_results_df


In [ ]:
# Hyperparameter tuning example: GridSearchCV on Random Forest Regressor
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_param_grid,
                        cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print('Best params:', rf_grid.best_params_)
best_rf_reg = rf_grid.best_estimator_
evaluate_regressor('Tuned Random Forest Regressor', best_rf_reg, X_train, X_test, y_train, y_test)


In [ ]:
# Compare regression model performance
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='R2', data=regression_results_df)
plt.title('Regression Model Comparison (R2 Score)')
plt.xticks(rotation=20)
plt.show()


## 8. Classification Techniques (3 Techniques, 3 Performance Measures)

We convert `delivery_duration` into speed categories (**Fast / Medium / Slow**) using tertiles, then classify using **Logistic Regression**, **Random Forest Classifier**, and **K-Nearest Neighbors**, tuned with GridSearchCV/RandomizedSearchCV.


In [ ]:
# Create classification target: bin delivery_duration into 3 classes
df['duration_class'] = pd.qcut(df['delivery_duration'], q=3, labels=['Fast', 'Medium', 'Slow'])
print(df['duration_class'].value_counts())


In [ ]:
Xc = df[selected_features]
yc = df['duration_class']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

scaler_c = StandardScaler()
Xc_train_scaled = scaler_c.fit_transform(Xc_train)
Xc_test_scaled = scaler_c.transform(Xc_test)


In [ ]:
def evaluate_classifier(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average='weighted')
    rec = recall_score(y_te, preds, average='weighted')
    f1 = f1_score(y_te, preds, average='weighted')
    print(f'--- {name} ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(classification_report(y_te, preds))
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

clf_results = []
clf_results.append(evaluate_classifier('Logistic Regression', LogisticRegression(max_iter=1000),
                                        Xc_train_scaled, Xc_test_scaled, yc_train, yc_test))
clf_results.append(evaluate_classifier('Random Forest Classifier', RandomForestClassifier(n_estimators=100, random_state=42),
                                        Xc_train, Xc_test, yc_train, yc_test))
clf_results.append(evaluate_classifier('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=5),
                                        Xc_train_scaled, Xc_test_scaled, yc_train, yc_test))

classification_results_df = pd.DataFrame(clf_results)
classification_results_df


In [ ]:
# Hyperparameter tuning example: RandomizedSearchCV on Random Forest Classifier
rf_clf_param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

rf_clf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_clf_param_dist,
                                    n_iter=8, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)
rf_clf_search.fit(Xc_train, yc_train)

print('Best params:', rf_clf_search.best_params_)
best_rf_clf = rf_clf_search.best_estimator_
evaluate_classifier('Tuned Random Forest Classifier', best_rf_clf, Xc_train, Xc_test, yc_train, yc_test)


In [ ]:
# Confusion matrix for best classifier
preds = best_rf_clf.predict(Xc_test)
cm = confusion_matrix(yc_test, preds, labels=['Fast', 'Medium', 'Slow'])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fast', 'Medium', 'Slow'],
            yticklabels=['Fast', 'Medium', 'Slow'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Tuned Random Forest Classifier')
plt.show()


In [ ]:
# Compare classification model performance
classification_results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1']].plot(
    kind='bar', figsize=(10, 6))
plt.title('Classification Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Conclusion

- Summarize which regression model achieved the lowest MAE/RMSE and highest R2.
- Summarize which classification model achieved the best accuracy/F1-score after tuning.
- Discuss which features were most important for predicting ETA.
- Note any limitations (e.g., data noise, missing real-time traffic/weather data) and possible future improvements (e.g., adding weather/traffic data, deep learning models, real-time features).
